# Practical 5: Transfer Learning via Frozen Feature Extraction
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit III (Transfer Learning & Fine-Tuning) | **CO:** CO3  
**Problem Statement Context:** SIH26131 — Rapid Crop Disease Classification with Minimal Compute  

---

## 1. Inductive Transfer Learning Theory
Training deep vision models from scratch requires millions of annotated samples. In agriculture, annotated disease datasets are often small ($10^2$ to $10^4$ images).
**Feature Extraction** leverages knowledge encoded in backbones pre-trained on generic large-scale corpora (ImageNet):
1. **Freeze Feature Extractor:** Fix all convolutional layer weights ($	heta_{backbone}$ where $
abla_{	heta} = 0$).
2. **Train Linear Classifier:** Attach a customized linear classification head ($	heta_{head}$) and optimize only these parameters on the crop dataset:

$$\min_{	heta_{head}} rac{1}{N} \sum_{i=1}^N \mathcal{L}_{CE}\left( f(x_i; 	heta_{backbone}^{frozen}, 	heta_{head}), y_i ight)$$

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.dataset import PlantDiseaseClassificationDataset
from src.transforms import get_classification_transforms
from src.models_cnn import build_transfer_classifier

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"[✓] Active Device: {device}")

## 2. Load PlantVillage Crop Dataset

In [ ]:
data_dir = REPO_ROOT / "data" / "samples" / "classification" / "lab_plantvillage"
transforms = get_classification_transforms(image_size=(224, 224), is_training=False)

dataset = PlantDiseaseClassificationDataset(root_dir=data_dir, transform=transforms)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

print(f"[✓] Loaded {len(dataset)} crop leaf samples across {len(dataset.classes)} classes:")
print(dataset.classes)

## 3. Freeze Backbone & Train Head

In [ ]:
model = build_transfer_classifier(
    backbone_name="resnet18",
    num_classes=len(dataset.classes),
    freeze_backbone=True
).to(device)

# Verify only head parameters have requires_grad=True
trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Total Parameter Tensors: {len(list(model.parameters()))}")
print(f"Trainable Head Tensors : {len(trainable_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(trainable_params, lr=1e-3)

model.train()
for epoch in range(3):
    total_loss = 0.0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/3] - Loss: {total_loss / len(dataloader):.4f}")

## 4. Evaluation & Confusion Matrix

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in dataloader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.matshow(cm, cmap="Blues")
fig.colorbar(cax)
ax.set_title("Crop Disease Confusion Matrix", pad=20)
ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")
plt.tight_layout()
plt.show()

## 5. Practical Takeaways
- Feature extraction trains significantly faster because backpropagation stops at the final classification layer without propagating gradients through dozens of deep convolutional blocks.
- Suitable for initial baseline models and lightweight IoT / edge deployment.